# Skill Builder Lab — Event-Driven Automation Systems
## Atlas Smart Research Request & Escalation System



> **Important:** Colab contains all instructions and starter code. Atlas and n8n both run on **your computer**, not inside Colab. Code cells marked **COPY TO VS CODE** are starter templates and are not expected to work inside Colab until implemented.

## Business Scenario

Northstar Systems already uses **Atlas** to analyze uploaded company PDF, TXT, CSV and Excel files. In Lesson 3, Atlas generated a fixed research report and sent it to n8n for email delivery. The company now needs a separate **research request and priority escalation** feature.

Employees must be able to:

1. Choose the company or companies whose files are already uploaded to Atlas.
2. Enter a **custom research question** and recipient email.
3. Select a **Normal** or **Urgent** priority.
4. Ask Atlas to generate a new research answer from the uploaded documents/tables.
5. Send the answer to a new **n8n webhook** as JSON.
6. Receive an appropriately formatted email according to the priority.
7. See a success or meaningful error message in Atlas.

**Do not rebuild Atlas's RAG backend.** Reuse the existing `answer_question()` function and existing company-file samples. This Skill Builder assesses event-driven integration and workflow handling.

## Learning Outcomes

By completing this Skill Builder you will be able to:

- Reuse an existing RAG backend from a new Gradio callback.
- Validate user input and assemble a predictable JSON request.
- Send an HTTP POST request to a locally running n8n webhook and interpret a JSON response.
- Build an event-driven n8n workflow using **Webhook → If → Basic LLM Chain → Markdown → If (priority) → Send Email → Respond to Webhook**.
- Configure validation, separate Normal/Urgent email routes, and error outputs.
- Test the Python helpers separately from the live workflow, inspect n8n Executions, and document results.
- Export and submit a secure, reproducible project on GitHub.

### Tools and boundaries

| Tool | Used for |
|---|---|
| **Google Colab** | Read the assignment and copy incomplete code/test templates |
| **VS Code + local Python 3.11/3.12** | Extend Atlas and run Python tests |
| **Local n8n (npm, no Docker)** | Configure conditional email workflow and export workflow JSON |
| **GitHub** | Submit notebook, Python files, workflow export and evidence |

**Not required:** a new vector database, a new RAG pipeline, Docker, cloud deployment, n8n Execute Command, or a Schedule Trigger. Email sending needs an SMTP account; report generation uses the model connection taught in Lesson 3.

## Provided Resources

- **Atlas project repository:** https://github.com/almabetter-school-ds/Atlas_AI_Company_Assistant
- **Existing files:** `app.py`, `app_n8n.py`, `backend.py`, `requirements.txt` and `sample_data/` (verify the files after downloading the repository).
- **Starter assets in this assignment package:** `automation_helpers_starter.py`, `test_automation_contract.py`, `.gitignore.example`, `README_template.md`.
- **n8n documentation:** https://docs.n8n.io/ (Webhook, If, Basic LLM Chain, Markdown, Send Email and Respond to Webhook).

**Credentials:** You need a DeepSeek key as in the lessons and an SMTP account configured **inside n8n**. Keep `.env`, passwords, app passwords and credentials out of GitHub. Work only with the fictional financial sample files supplied with Atlas; do not upload personal or confidential reports.

**Important local-network rule:** n8n at `http://localhost:5678` refers to your own computer. Google Colab runs remotely and cannot directly access n8n on your laptop through `localhost`. Therefore, copy the Python starter code into VS Code and perform integrated tests there.

## Submission Instructions

1. Finish each step in this notebook and write short answers in its reflection sections.
2. Implement the incomplete Python functions in `automation_helpers.py` on your local computer.
3. Create `app_automation.py` from the existing `app_n8n.py`, leaving the original application and `backend.py` unchanged.
4. Build **your own** n8n workflow and export it as `n8n_workflow/atlas_priority_workflow.json`.
5. Capture screenshots that prove successful Normal/Urgent requests and valid error handling.
6. Commit your completed Skill Builder notebook, Python changes, sanitized workflow export, selected screenshots and `README.md` to one GitHub repository.
7. Submit **only the repository URL**, and make the repository visible to your evaluator.

Your final repository must be safe to share: **no `.env`, real credentials, access tokens, model keys, or private employee data**.

## Part A — Prepare Atlas in VS Code
### Step 0. Set up the original Atlas project

**Where:** VS Code terminal on your computer. **Do not execute these commands in Colab.**

1. Download or clone the provided Atlas repository.
2. Open the extracted project in VS Code and confirm `app_n8n.py`, `backend.py` and `sample_data/` exist.
3. Use Python 3.11 or 3.12 and install `requirements.txt` in a virtual environment.
4. Configure `.env` using your own DeepSeek key as in Lesson 1. Keep it in `.gitignore`.
5. Run `app_n8n.py` and confirm that you can upload a company document/table and ask a question. This is your baseline.

**Windows (PowerShell):**
```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
.\.venv\Scripts\python.exe app_n8n.py
```

**macOS/Linux:**
```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python app_n8n.py
```

**Expected result:** Atlas opens locally (usually `http://127.0.0.1:7860`) and can use uploaded sample files. Stop the baseline application before starting the new application, to avoid a port conflict.

**Purpose of Step:** This establishes a working Atlas baseline. The assignment changes only the automation layer, so you can distinguish new integration problems from existing RAG or environment problems.

### Step 1. Create separate files for your automation extension

**Where:** VS Code project folder.

1. Copy `app_n8n.py` and name the new copy **`app_automation.py`**.
2. Make a new file **`automation_helpers.py`** and copy the unfinished functions from Parts B and C into it. You may start from the supplied `automation_helpers_starter.py` (rename it).
3. Keep `backend.py` unchanged. Do not remove the existing Gradio upload and chatbot functions from `app_automation.py`.
4. Set the **new webhook path** to `atlas-priority` (not the existing `atlas-report` used in Lesson 3).
5. Choose a local configuration mechanism; a simple environment variable is sufficient:

```python
import os
N8N_WEBHOOK_URL = os.getenv(
    "N8N_WEBHOOK_URL",
    "http://localhost:5678/webhook/atlas-priority"
)
```

**Expected result:** Both the original Lesson 3 app and your new extension exist, with separate webhook paths.

**Purpose of Step:** Keeping a separate application and webhook allows you to develop and debug the new priority-routing feature without overwriting the working Lesson 3 solution.

### Step 2. Trace the existing research call before modifying the interface

**Where:** Read `app_n8n.py` and `backend.py` in VS Code.

Locate the existing `send_email_report(...)` function and the `answer_question(...)` call. Confirm the parameter order:

```python
answer_question(key, question, companies, source, store_type, rerank, hybrid)
```

For this assignment, call the existing backend with **the user's custom question** and `source="Both"`:

```python
answer_question(key, custom_question, companies, "Both", store_type, rerank, hybrid)
```

**Do not call DeepSeek a second time in Python just to create the research answer.** The existing backend handles research. The n8n Basic LLM Chain will **format** the report as taught in Lesson 3.

**Student check:** Write down the callback and Gradio component variables already used for `key`, `companies`, `store_type`, `rerank` and `hybrid` before wiring the new button.

**Purpose of Step:** This confirms the correct interface to the already-built Atlas RAG backend and prevents duplicating the retrieval/analysis logic in the new assignment.

## Part B — Complete the Python Automation Helpers

**Where:** The function templates appear in Colab for guidance, but must be completed in **`automation_helpers.py` in VS Code**. Do not replace the placeholders in the provided notebook with an entire solved application; keep the same function signatures and document how you tested them.

### Step 3. Validate and normalize the research request

**Your task:** Complete `validate_research_request(recipient, companies, question, priority)`.

- Strip surrounding whitespace from the recipient and question.
- Accept only **`normal`** or **`urgent`** (case-insensitive; return lowercase).
- Check a basic email-address format (not a guarantee of deliverability).
- Require at least one company and a non-empty question; normalize company names into a list of strings.
- Raise `ValueError` with a helpful message for invalid values.
- Return `{'recipient': ..., 'companies': [...], 'question': ..., 'priority': ...}`.

**Example:** `(' analyst@example.com ', ['Apple'], 'Compare FY2024 revenue', 'Urgent')` → normalized dictionary with `priority='urgent'`.

**Hint:** Use `str.strip()`, a basic regular expression and membership in a set.

**Purpose of Code:** The function prevents invalid requests from triggering costly RAG processing or entering the email workflow. Normalization also ensures Python and n8n agree on the meaning of the priority field.

In [ ]:
# COMPLETED: automation_helpers.py (copied from the VS Code project)
import re

VALID_PRIORITIES = {"normal", "urgent"}
EMAIL_PATTERN = re.compile(r"^[^\s@]+@[^\s@]+\.[^\s@]+$")
MAX_QUESTION_CHARS = 2000


def validate_research_request(
    recipient: str,
    companies: list[str],
    question: str,
    priority: str,
) -> dict:
    """Return normalized input fields or raise ValueError."""
    recipient = (recipient or "").strip()
    question = (question or "").strip()
    priority = (priority or "").strip().lower()

    if not recipient or not EMAIL_PATTERN.match(recipient):
        raise ValueError("Enter a valid recipient email address.")

    if isinstance(companies, str):  # tolerate a single company name
        companies = [companies]
    cleaned = [str(c).strip() for c in (companies or []) if c is not None and str(c).strip()]
    if not cleaned:
        raise ValueError("Select at least one company.")

    if not question:
        raise ValueError("Enter a research question.")
    if len(question) > MAX_QUESTION_CHARS:
        raise ValueError(f"Keep the research question within {MAX_QUESTION_CHARS:,} characters.")

    if priority not in VALID_PRIORITIES:
        raise ValueError("Priority must be 'normal' or 'urgent'.")

    return {"recipient": recipient, "companies": cleaned, "question": question, "priority": priority}

### Step 4. Extract a usable research answer from Atlas

**Your task:** Complete `extract_research_text(research_result)`.

- Read the `answer` field from the dictionary returned by Atlas.
- Check that it is a non-empty string.
- Check that at least one of `documents` or `calculations` is non-empty; these fields contain the evidence returned by Atlas.
- If there is no answer or no evidence, raise `ValueError` instead of sending an unsupported email.
- Return the clean answer text.

**Expected input:** `{'answer': '...', 'documents': [...], 'calculations': [...]}`.

**Important:** An evidence object is not a proof that every statement in the generated answer is correct. Students must still inspect the generated result against the uploaded files.

**Purpose of Code:** This separates research-answer validation from HTTP sending and blocks empty or obviously unsupported results before they become automated email reports.

In [ ]:
# COMPLETED: automation_helpers.py (copied from the VS Code project)
def extract_research_text(research_result: dict) -> str:
    """Return a non-empty Atlas answer supported by at least one evidence item."""
    if not isinstance(research_result, dict):
        raise ValueError("Atlas returned an unexpected result.")
    answer = research_result.get("answer")
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("Atlas could not generate a research answer.")
    if not research_result.get("documents") and not research_result.get("calculations"):
        raise ValueError("Atlas found no supporting evidence in the selected files, so no email was sent.")
    return answer.strip()

### Step 5. Build the JSON payload for n8n

**Your task:** Complete `build_webhook_payload(validated, report)`.

Your return dictionary must have **exactly these five fields**, so n8n expressions remain consistent:

```json
{
  "recipient": "analyst@example.com",
  "company": "Apple, Microsoft",
  "question": "Compare FY2024 revenue for these companies.",
  "priority": "urgent",
  "report": "Research generated by Atlas from the uploaded files..."
}
```

- Join company names into a comma-separated string for `company`.
- Copy the normalized recipient/question/priority.
- Check that `report` is non-empty; raise `ValueError` otherwise.
- Do **not** include API keys, SMTP credentials, local file paths or prior chat messages.

**Purpose of Code:** A stable five-field JSON contract is the boundary between the Python application and the n8n workflow. Both sides must agree on field names and values.

In [ ]:
# COMPLETED: automation_helpers.py (copied from the VS Code project)
def build_webhook_payload(validated: dict, report: str) -> dict:
    """Return recipient, company, question, priority and report for n8n."""
    if not isinstance(report, str) or not report.strip():
        raise ValueError("The research report is empty.")
    return {
        "recipient": validated["recipient"],
        "company": ", ".join(validated["companies"]),
        "question": validated["question"],
        "priority": validated["priority"],
        "report": report.strip(),
    }

### Step 6. Send the request to n8n and interpret HTTP/JSON

**Your task:** Complete `post_to_n8n(webhook_url, payload, timeout=180)` using `requests.post`.

- Send `json=payload` and an explicit timeout.
- Catch **connection errors and timeouts** (`requests.RequestException`) and return an error dictionary, rather than crashing the Gradio interface.
- Parse the n8n JSON reply. Handle a non-JSON reply as an error.
- Do not assume HTTP 200 means success: inspect the returned `status` field.
- Normalize the result to:

```python
{
    'http_status': 200,          # or 400, 502, 0 (no HTTP reply)
    'status': 'success',         # or 'error'
    'message': '...',
    'priority': 'normal'         # if provided by workflow
}
```

- On any HTTP error (e.g., 400 or 502), return `status='error'` and a meaningful message. Never print a secret or the complete request body in an error.

**Hint:** `requests.post(...)`, `response.status_code`, `response.json()` and `response.ok` were introduced in Lesson 3.

**Purpose of Code:** This function provides the actual event-driven link from Atlas to n8n and ensures network failures, invalid input responses and email failures can be displayed safely to the user.

In [ ]:
# COMPLETED: automation_helpers.py (copied from the VS Code project)
import requests


def _error(http_status: int, message: str) -> dict:
    return {"http_status": http_status, "status": "error", "message": message}


def post_to_n8n(
    webhook_url: str,
    payload: dict,
    timeout: int = 180,
) -> dict:
    """POST JSON and return a normalized HTTP/workflow result dict.

    Never raises for network/HTTP problems and never echoes the payload.
    """
    try:
        response = requests.post(webhook_url, json=payload, timeout=timeout)
    except requests.exceptions.Timeout:
        return _error(0, f"n8n did not respond within {timeout} seconds.")
    except requests.exceptions.ConnectionError:
        return _error(0, "Could not connect to n8n. Is n8n running and the workflow published?")
    except requests.RequestException as exc:
        return _error(0, f"Request to n8n failed ({type(exc).__name__}).")

    http_status = response.status_code
    try:
        body = response.json()
    except ValueError:
        body = None

    if not isinstance(body, dict):
        if not response.ok:
            return _error(http_status, f"n8n returned HTTP {http_status} without a valid JSON reply.")
        return _error(http_status, "n8n returned a reply that is not valid JSON.")

    message = str(body.get("message") or "").strip()
    wf_status = str(body.get("status") or "").strip().lower()

    if not response.ok:
        return _error(http_status, message or f"n8n returned HTTP {http_status}.")
    if wf_status != "success":  # HTTP 200 alone is not success
        return _error(http_status, message or "The n8n workflow did not report success.")

    result = {"http_status": http_status, "status": "success",
              "message": message or "Research request submitted."}
    if body.get("priority"):
        result["priority"] = str(body["priority"])
    return result

### Step 7. Convert a workflow result into a readable Atlas status

**Your task:** Complete `format_automation_status(result)`.

- For HTTP success with `result['status'] == 'success'`, return a message beginning with **`Success:`**.
- Otherwise return a message beginning with **`Error:`**.
- Include the workflow message when present.
- Do not show the raw payload or a server traceback in the Gradio UI.

**Examples:**
- `{'http_status': 200, 'status': 'success', 'message': 'Urgent alert sent.'}` → `Success: Urgent alert sent.`
- `{'http_status': 400, 'status': 'error', 'message': 'Invalid input.'}` → `Error: Invalid input.`

**Purpose of Code:** This translates technical responses into user-friendly feedback and prevents the interface from incorrectly displaying success after an HTTP or workflow failure.

In [ ]:
# COMPLETED: automation_helpers.py (copied from the VS Code project)
def format_automation_status(result: dict) -> str:
    """Return a safe user-facing Success: or Error: message."""
    result = result if isinstance(result, dict) else {}
    http_status = result.get("http_status")
    ok_http = isinstance(http_status, int) and 200 <= http_status < 300
    message = str(result.get("message") or "").strip()
    if ok_http and result.get("status") == "success":
        return "Success: " + (message or "Research request submitted.")
    return "Error: " + (message or "The research request could not be completed.")

### Step 8. Orchestrate RAG research and event-driven automation

**Your task:** Complete `execute_research_automation(...)`.

1. Call `validate_research_request` **before** generating any research.
2. Call the existing Atlas `answer_question` with the user question and `source='Both'`.
3. Call `extract_research_text`, then `build_webhook_payload`.
4. Call `post_to_n8n` and `format_automation_status`.
5. Return a readable error for known input, backend, or communication failures; avoid exposing sensitive details.

**Do not:** call Atlas using `FIXED_RESEARCH_QUESTION`, read previous chatbot history, add new vector stores, or change `backend.py`.

**Signature:** `execute_research_automation(key, recipient, companies, question, priority, store_type, rerank, hybrid, webhook_url) -> str`.

**Hint:** The existing backend API from Lesson 1 is `answer_question(key, question, companies, 'Both', store_type, rerank, hybrid)`.

**Purpose of Code:** This is the complete new application flow: validate the employee's request, reuse Atlas RAG, send the resulting research as an event to n8n, and return the automation outcome to Gradio.

In [ ]:
# COMPLETED: automation_helpers.py (copied from the VS Code project)
# answer_question is imported from backend.py in automation_helpers.py

def execute_research_automation(
    key,
    recipient: str,
    companies: list[str],
    question: str,
    priority: str,
    store_type: str,
    rerank: bool,
    hybrid: bool,
    webhook_url: str,
) -> str:
    """Validate -> Atlas answer_question -> payload -> webhook -> UI status."""
    try:
        validated = validate_research_request(recipient, companies, question, priority)
    except ValueError as exc:
        return f"Error: {exc}"

    try:
        research = answer_question(
            key, validated["question"], validated["companies"], "Both",
            store_type, rerank, hybrid,
        )
        report = extract_research_text(research)
        payload = build_webhook_payload(validated, report)
    except ValueError as exc:  # expected: bad workspace, no evidence, empty answer...
        return f"Error: {exc}"
    except Exception:
        logger.exception("Atlas research generation failed")
        return "Error: Atlas could not generate the research. See the terminal log for details."

    result = post_to_n8n(webhook_url, payload)
    return format_automation_status(result)

## Part C — Extend the Gradio Interface in VS Code
### Step 9. Add the research request controls and connect the new callback

**Where:** `app_automation.py`, created from `app_n8n.py`.

1. Import your completed helper function, and configure the new webhook URL from an environment variable.
2. Add a **Custom Research Question** `gr.Textbox` (multiline, not a fixed question).
3. Add **Priority** `gr.Dropdown(choices=['normal', 'urgent'], value='normal')`.
4. Keep the existing company, recipient, vector-store, BM25 and rerank controls; avoid duplicating them if already present.
5. Add a **Send Research Request** button and a read-only **Automation Status** textbox.
6. Bind the new button to `automation_ui(...)`, forwarding the correct existing workspace and settings components. Inspect the actual variable names in your copy of `app_n8n.py`; do not guess them.
7. Leave the original chat and existing email-report button available if possible.

The starter below is a callback **template**, not a complete replacement for the existing Gradio app. Paste it in a suitable location in `app_automation.py` and fill the TODOs.

**Expected result:** After uploading company files, a user can enter any research question, choose `normal`/`urgent`, and receive the workflow status in the UI.

**Purpose of Code:** This connects user interaction to the new event-driven orchestration function while preserving the existing company-document upload, RAG research and settings functionality.

In [ ]:
# COMPLETED: app_automation.py (excerpt; the full file is in the repository)
import os
from automation_helpers import execute_research_automation

N8N_WEBHOOK_URL = os.getenv("N8N_WEBHOOK_URL", "http://localhost:5678/webhook/atlas-priority")


def automation_ui(key, recipient, companies, question, priority,
                  store_type, rerank, hybrid):
    """Gradio callback for the new priority research workflow."""
    webhook_url = os.getenv("N8N_WEBHOOK_URL", N8N_WEBHOOK_URL)
    return execute_research_automation(
        key, recipient, companies, question, priority,
        store_type, rerank, hybrid, webhook_url,
    )

# Layout additions (inside the right-hand column of app_n8n.py's Blocks):
#   research_question = gr.Textbox(label="Custom Research Question", lines=3)
#   priority = gr.Dropdown(choices=["normal", "urgent"], value="normal", label="Priority")
#   research_button = gr.Button("Send Research Request", variant="primary")
#   automation_status = gr.Textbox(label="Automation Status", interactive=False)
# Wiring (real component names from app_n8n.py: key, recipient_email, selected,
# store_type, rerank, hybrid):
#   research_button.click(fn=automation_ui,
#       inputs=[key, recipient_email, selected, research_question, priority,
#               store_type, rerank, hybrid],
#       outputs=automation_status, **options)

## Part D — Build the n8n Priority-Routing Workflow
### Step 10. Create a NEW webhook workflow

**Where:** Local n8n interface at `http://localhost:5678` (run `n8n start`, as in Lesson 2).

1. Create a workflow called **Atlas Priority Research Automation**.
2. Add a **Webhook** trigger with HTTP method `POST`, path `atlas-priority`, authentication `None` for **localhost-only coursework**, and **Respond = Using 'Respond to Webhook' Node**.
3. Use the **Test URL** while building (`/webhook-test/atlas-priority`) and click **Listen for test event** before each test; use the **Production URL** after publishing (`/webhook/atlas-priority`).
4. Send the five-field sample payload below from your **local VS Code terminal**, or use an HTTP testing client. Do not send the request from remote Colab to localhost.
5. Open the Webhook node execution data and verify fields arrive under `$json.body`.

**Expected received fields:** `recipient`, `company`, `question`, `priority`, `report`.

**Security:** No authentication is acceptable only for this local learning exercise. Do not expose this webhook to the public Internet. A deployed system requires authentication and access controls.

**Purpose of Workflow:** The Webhook is the event trigger that receives Atlas research requests and starts the n8n execution whenever Python sends an HTTP POST request.

In [ ]:
# SAMPLE TEST PAYLOAD — run this as a local Python script in VS Code, not Colab.
# This is a provided test fixture, NOT a completed workflow.
import requests

SAMPLE_PAYLOAD = {
    "recipient": "your-own-test-address@example.com",  # replace locally
    "company": "Apple, Microsoft",
    "question": "Compare FY2024 revenue and reporting dates.",
    "priority": "normal",
    "report": "Apple and Microsoft have different fiscal year-end dates. "
              "Use only verified facts from the uploaded data.",
}

# TODO: When the n8n Webhook node is listening for a test event,
# uncomment and adapt the following lines in a LOCAL script:
# response = requests.post(
#     "http://localhost:5678/webhook-test/atlas-priority",
#     json=SAMPLE_PAYLOAD, timeout=180,
# )
# print(response.status_code, response.text)

### Step 11. Validate the request in an If node

**Where:** n8n canvas.

1. Connect **Webhook → If (Validate Research Request)**.
2. Configure the If node to accept only requests for which all conditions hold:
   - recipient looks like an email address;
   - `company`, `question` and `report` contain non-empty text;
   - `priority` is exactly `normal` or `urgent`.
3. In the true output, continue to report formatting.
4. In the false output, connect **Respond to Webhook (Invalid Request)** with HTTP **400** and the JSON below.
5. Test a valid request and an invalid-priority request directly against the webhook. The invalid request must not reach any LLM or email node.

```json
{"status": "error", "message": "Invalid recipient, company, question, priority, or report."}
```

**Hints:** Use n8n expressions like `{{ $json.body.priority }}` and Boolean conditions. For this coursework, simple regex/blank checks are sufficient; real systems need stronger authentication and validation. Python validation helps the user, but **webhook-side validation remains mandatory** because a webhook can be called directly.

**Purpose of Workflow:** This creates a reliable validation gate at the receiving system and protects the email and LLM nodes from malformed requests that bypass the Atlas UI.

### Step 12. Format the Atlas research into a fixed email report

**Where:** n8n canvas, on the **valid** branch.

1. Add **Basic LLM Chain** and connect the **DeepSeek Chat Model** credential as demonstrated in Lesson 3.
2. Give the chain a **fixed report-formatting prompt**. Its input research must come from the webhook's `report` field; it should not retrieve files again.
3. Request six sections: (1) Executive Summary, (2) Companies & Reporting Periods, (3) Financial Performance, (4) Comparisons, (5) Key Findings, and (6) Limitations.
4. Preserve financial amounts, currencies and period-end dates exactly as received. Never manufacture missing figures. Avoid internal evidence markers such as `[A1]` in the email body.
5. Configure the LLM node's **error output** to return a JSON **502** via **Respond to Webhook (Report Error)**.
6. Use the same **Markdown** node conversion from Lesson 3 to turn the report into HTML before sending email.

**Helpful expression:** `{{ $('Webhook').item.json.body.report }}`.

**Expected result:** One HTML report created from the incoming research, irrespective of request priority.

**Purpose of Workflow:** The LLM chain standardizes how the research is presented while keeping Atlas responsible for retrieval. The Markdown node converts that structured text to email-friendly HTML.

### Step 13. Route the request by priority using a second If node

**Where:** n8n canvas **after** Markdown conversion.

1. Add **If (Is Urgent?)** and check whether the webhook's `priority` equals `urgent`.
2. On **True**, route to **Send Urgent Research Email**.
3. On **False**, route to **Send Normal Research Email**.
4. Both branches must use the **same generated HTML report** and the recipient supplied by the Webhook.
5. Give both branches a distinct subject, for example:

| Route | Example subject |
|---|---|
| Urgent | `URGENT | Atlas Research | Apple, Microsoft` |
| Normal | `Atlas Research | Apple, Microsoft` |

**Hint:** The earlier validation node already prevented unsupported priorities. Reference the original Webhook data when a later node has changed `$json`.

**Expected result:** Normal requests go to the Normal email node; urgent requests go to the Urgent email node. Only **one** email branch should run per request.

**Purpose of Workflow:** Conditional routing turns one generic workflow into a priority-aware automation, a new behavior not present in the original fixed-report email exercise.

### Step 14. Configure email delivery and explicit webhook responses

**Where:** n8n canvas.

1. Configure **both** Send Email nodes using the SMTP account setup taught in Lesson 3. Use your own **test email address** as recipient while testing.
2. For each email node, select **HTML** format and insert the Markdown node's HTML output. Keep the layout simple and readable.
3. Under **Settings → On Error**, choose **Continue (using error output)**. Connect successful send output to a JSON HTTP **200** response and email error output to a JSON HTTP **502** response.
4. Distinguish `normal` and `urgent` in each successful response. A successful SMTP handoff is not a guarantee of final inbox delivery.
5. Ensure *every executable branch* reaches its own **Respond to Webhook** node. For each individual request, **exactly one** response should be sent.

**Required response contract:**

| Scenario | HTTP | Response body |
|---|---|---|
| Normal email submitted | 200 | `{"status":"success","message":"Normal research email submitted.","priority":"normal"}` |
| Urgent email submitted | 200 | `{"status":"success","message":"Urgent research alert submitted.","priority":"urgent"}` |
| Invalid payload | 400 | `{"status":"error","message":"Invalid recipient, company, question, priority, or report."}` |
| LLM/report generation failure | 502 | `{"status":"error","message":"Report generation failed."}` |
| SMTP/email failure | 502 | `{"status":"error","message":"Email sending failed."}` |

**Note:** Avoid returning SMTP credentials, full stack traces or uploaded research content in error JSON.

**Purpose of Workflow:** Explicit success and failure responses let Atlas show the correct outcome. Separate error outputs prevent an unsuccessful email step from being reported as a success.

### Step 15. Test, publish and export the n8n workflow

**Where:** n8n.

1. Inspect each node connection, especially the false-validation branch, both priority routes, the LLM error output and both email error outputs.
2. Test first using the **Test URL** while listening, then **Save and Publish** the workflow (terminology varies slightly by n8n version).
3. Verify the **Production URL** is `http://localhost:5678/webhook/atlas-priority`, or copy the URL shown in your instance.
4. Set `N8N_WEBHOOK_URL` in your local Atlas configuration to the production URL. Do not hard-code private domains or credentials into the repository.
5. Trigger a normal request and an urgent request through the Atlas UI, and review **n8n → Executions**.
6. Export the **student-created workflow** as JSON. Save the file in `n8n_workflow/atlas_priority_workflow.json`.
7. Inspect the export before committing. It must contain **no API keys, SMTP passwords, headers with secrets, or personal email content**. Credential references may still need to be configured when another student imports it.

**Expected result:** A published and working local webhook, exported workflow JSON, and proof of two different email branches.

**Purpose of Workflow:** Publishing makes the production endpoint available; execution history and workflow export provide reproducible evidence of the event-driven system students built.

## Part E — Test the Python Helpers and the Full Automation
### Step 16. Run the provided contract tests locally

**Where:** VS Code terminal. Download the supplied `test_automation_contract.py` into the same folder as `automation_helpers.py`.

1. Complete Steps 3–8 first.
2. Run `python -m unittest -v test_automation_contract.py` with your virtual environment's Python.
3. Fix any failing tests and repeat. The tests mock backend and webhook calls; **no live DeepSeek, SMTP or n8n service is required**.
4. Do not change the provided assertions just to make incomplete functions pass.
5. Record test results in the table after this step.

**Important:** Some tests check the external contract, while manual testing is still required for UI wiring and real workflow behavior.

**Purpose of Code:** Isolated tests make sure request validation, payload shape, response handling and orchestration behave correctly before students troubleshoot a live system involving multiple services.

In [ ]:
# Command (run in the project folder):
#   python -m unittest -v test_automation_contract.py
# Result of the run on the delivered code (26 tests, mocked backend + local stub HTTP server):
local_test_results = {
    "tests_run": 26,
    "tests_passed": 26,
    "tests_failed": 0,
    "notes": "All pass. Covers invalid input (recipient/companies/question/priority), "
             "HTTP 400/502, HTTP 200 with status=error, non-JSON replies, timeout, "
             "and connection failure (closed port = n8n stopped). "
             "Re-run on your own machine and update these numbers.",
}
local_test_results

### Step 17. Verify the complete workflow with a fixed test matrix

**Where:** Atlas UI in VS Code + n8n Executions + your own test inbox.

Run **all six** scenarios. For a truly invalid webhook test, call the local webhook directly; Atlas may reject it first.

| Test | Input / condition | Expected behavior | Evidence |
|---|---|---|---|
| T1 | Valid research request, `normal` | Normal subject; workflow HTTP 200 | UI + email + n8n execution |
| T2 | Valid research request, `urgent` | Urgent subject; workflow HTTP 200 | UI + email + n8n execution |
| T3 | Invalid email entered in Atlas | UI displays `Error:`; no webhook call | UI screenshot |
| T4 | Invalid priority sent **directly** to n8n | HTTP 400; no LLM/email action | HTTP reply + execution |
| T5 | Empty question in Atlas | UI displays `Error:` before RAG call | UI screenshot |
| T6 | n8n stopped when Atlas sends request | Connection error shown, not success | UI screenshot |

**Optional:** Test the SMTP-error branch using a separate **local test copy** of your workflow with a deliberately nonworking test SMTP setup; restore the working configuration afterward. Do not publish or screenshot secrets.

**Do not:** claim success only because Atlas returned HTTP 200; check the workflow JSON status and confirm the correct branch executed. Do not send reports to someone else's inbox without consent.

**Purpose of Workflow:** These tests demonstrate that the automation works in normal conditions and fails predictably across validation, routing and network boundaries.

In [ ]:
# Honest status: 'actual' reflects what has been verified so far.
# Items marked LIVE need your running Atlas + n8n + SMTP inbox - run them and replace the text.
TEST_RESULTS = [
    {"id": "T1", "expected": "Normal email + HTTP 200", "actual": "LIVE - TODO: run in Atlas UI, attach screenshot/execution", "pass": None},
    {"id": "T2", "expected": "Urgent email + HTTP 200", "actual": "LIVE - TODO: run in Atlas UI, attach screenshot/execution", "pass": None},
    {"id": "T3", "expected": "UI validation error", "actual": "Unit test: 'Error: Enter a valid recipient email address.', no RAG/webhook call. LIVE UI screenshot still TODO", "pass": None},
    {"id": "T4", "expected": "HTTP 400 on direct invalid request", "actual": "If-node expressions simulated in Node (priority 'high' -> false). LIVE direct POST + execution still TODO", "pass": None},
    {"id": "T5", "expected": "Empty question rejected", "actual": "Unit test: 'Error: Enter a research question.', backend not called. LIVE UI screenshot still TODO", "pass": None},
    {"id": "T6", "expected": "Connection error when n8n stopped", "actual": "Unit test vs closed port: 'Error: Could not connect to n8n...'. LIVE UI screenshot still TODO", "pass": None},
]
TEST_RESULTS

### Step 18. Document your observations

> Rows marked **TODO (live)** need your real Atlas + n8n + SMTP run. Replace them with what you actually see; do not leave them as is when submitting.

| Observation | Your findings |
|---|---|
| Normal workflow result | TODO (live): expect HTTP 200, `Success: Normal research email submitted.`, subject `Atlas Research \| <companies>`, only *Send Normal Research Email* in the n8n execution. |
| Urgent workflow result | TODO (live): expect HTTP 200, `Success: Urgent research alert submitted.`, subject `URGENT \| Atlas Research \| <companies>`, only *Send Urgent Research Email* executed. |
| Invalid request handling | Verified offline: Python rejects bad recipient/empty question/priority before any RAG call (unit tests); the n8n If expressions reject bad priority/email/blank fields (Node simulation). TODO (live): direct POST returns HTTP 400 and no LLM/email node runs. |
| n8n stopped / network error | Verified offline: connecting to a closed port returns `Error: Could not connect to n8n...`, never `Success`. TODO (live): UI screenshot. |
| One issue you debugged | Example from this build: `python -I -m unittest` could not import the local test module because `-I` removes the current directory from `sys.path`; running without `-I` fixed it. Replace/extend with your own live issue. |

**Answers**

1. **What event starts the workflow?** An HTTP POST arriving at the n8n Webhook node (`/webhook/atlas-priority`) when a user clicks *Send Research Request* in Atlas. The Schedule Trigger in Lesson 2 starts runs by the clock whether or not anything happened; the webhook starts a run only when an external event occurs, so it is on-demand and carries data in the request.
2. **Why does `answer_question()` run in Atlas?** Atlas already owns the uploaded files, indexes, retrieval, reranking and validated table calculations in its session workspace. n8n has no access to them, and re-implementing retrieval would duplicate logic and risk inconsistent answers. n8n only formats and delivers the finished research.
3. **Why validate in both Python and n8n?** Python validation gives fast, friendly feedback and avoids costly RAG/LLM calls on bad input. But a webhook can be called by anything (curl, another script), bypassing Atlas, so n8n must independently enforce the contract before using the LLM or sending email.
4. **HTTP status vs JSON `status`?** The HTTP status reports whether the request/response transport succeeded (200, 400, 502). The JSON `status` is the workflow's own business outcome. They can disagree (e.g. HTTP 200 with `status: error`), so Atlas requires both a 2xx code and `status == "success"`.
5. **What does a successful SMTP send mean?** The SMTP server accepted the message for delivery. It does not guarantee inbox delivery, that it avoided spam filtering, bounced later, or that the recipient read it.
6. **Why `/webhook/` instead of `/webhook-test/`?** The test URL only works while the editor is *listening for a test event* (usually one request). The production URL is active for a published/active workflow and works continuously without the editor open, which is what a deployed Atlas needs.


## Part F — Publish Your Work on GitHub
### Step 19. Prepare the final repository

**Where:** VS Code and GitHub. You may use the provided `README_template.md` as a starting point.

Your repository should resemble:

```text
Atlas_Priority_Research_Automation/
├── app_automation.py
├── automation_helpers.py
├── backend.py
├── requirements.txt
├── .env.example
├── .gitignore
├── n8n_workflow/
│   └── atlas_priority_workflow.json
├── screenshots/
│   ├── workflow_overview.png
│   ├── normal_request.png
│   ├── urgent_request.png
│   └── invalid_request.png
├── Skill_Builder_Lab_Event_Driven_Automation_Completed.ipynb
└── README.md
```

Include other **required original Atlas files** (including `sample_data/` if needed for reproducibility), while obeying the original project's license. `app.py` and `app_n8n.py` may be retained if useful for comparison.

**README minimum:** project goal, architecture summary, local setup for Python and n8n, environment-variable placeholders, test commands, how to import the workflow JSON, test results, what changed from Lesson 3, and limitations.

**Security checklist:** keep `.env`, virtual environments, cache folders, n8n storage, SMTP passwords and DeepSeek keys out of commits; sanitize screenshots and workflow JSON. A `.gitignore` does not remove a secret that has already been committed.

**Purpose of Step:** A reproducible, secure repository lets reviewers inspect Python implementation, exported n8n workflow and evidence together, without needing access to students' credentials.

## Evaluation Rubric

| Criterion | Marks |
|---|---:|
| Atlas/Gradio interface extension and correct backend reuse | 15 |
| Python validation, JSON payload, HTTP request and response handling | 20 |
| n8n webhook, validation and conditional priority routing | 25 |
| Report formatting, email and success/error handling | 20 |
| Local contract tests and live scenario evidence | 10 |
| GitHub repository, README, reflection and secret hygiene | 10 |
| **Total** | **100** |

## Submission Checklist

- [ ] Atlas original `backend.py` reused; no new RAG pipeline required.
- [ ] `app_automation.py` accepts a custom question, recipient and priority.
- [ ] All six Python helper functions in Steps 3–8 completed.
- [ ] Local unit tests recorded, including at least one invalid-input and one connection-failure test.
- [ ] New `atlas-priority` n8n workflow built, published and exported (not supplied prebuilt).
- [ ] Workflow validates payload, routes urgent/normal requests, and returns 200/400/502 appropriately.
- [ ] At least six manual test outcomes documented; screenshots contain no secrets.
- [ ] Notebook reflection questions completed.
- [ ] GitHub repository includes app, helpers, exported workflow, notebook, README and evidence.
- [ ] `.env`, SMTP credentials, API keys, tokens and local n8n data are excluded.

